<a href="https://colab.research.google.com/github/PriyanshuChaudhary00/Ai/blob/main/basic_image_classification_with_pyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fashion MNIST Image Classification using PyTorch

This project demonstrates how to build an Artificial Neural Network
for classifying Fashion MNIST images using PyTorch.

Topics covered:

- Custom Dataset
- DataLoader
- Neural Networks
- Dropout
- Training Loop
- Evaluation
- Hyperparameter Optimization with Optuna

In [546]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader ,Dataset
import numpy as np

In [547]:
torch.manual_seed(42)

In [548]:
device = torch.device("cuda")

In [549]:
df = pd.read_csv("/content/fashion-mnist_train.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [550]:
x = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [551]:
y_train = y
x_train = x/255.0
x_train.shape

(60000, 784)

Optuna

In [552]:
class customDataset(Dataset):
  def __init__(self , features , labels):
    self.features = torch.tensor(features , dtype=torch.float32).to(device)
    self.labels = torch.tensor(labels , dtype=torch.long).to(device)
  def __len__(self):
    return self.features.shape[0]
  def __getitem__(self , index):
    return self.features[index] , self.labels[index]

In [553]:
dataset_train = customDataset(x_train , y_train)
dataLoaded_train = DataLoader(dataset_train , batch_size=128 , shuffle= True)

In [554]:
df = pd.read_csv("/content/fashion-mnist_test.csv")
df.head()
x = df.iloc[:,1:].values
y = df.iloc[:,0].values
y_test = y
x_test = x/255.0
x_test.shape

(10000, 784)

In [555]:
dataset_test = customDataset(x_test , y_test)
dataLoaded_test = DataLoader(dataset_test , batch_size=128 , shuffle= True)

In [556]:
class ANN(nn.Module):
  def __init__(self):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(784 , 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p = 0.3),

        nn.Linear(128 , 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p = 0.3),

        nn.Linear(64 , 10),
    )
  def forward(self , x):
    y_pred = self.model(x)
    return y_pred

In [557]:
class MyNN(nn.Module):
  def __init__(self , input_dim , output_dim , hidden_layers , neurons_per_layers , dropOut_rate):
    super().__init__()
    layers = []
    for i in range(hidden_layers):

      layers.append(nn.Linear(input_dim , neurons_per_layers))
      layers.append(nn.BatchNorm1d(neurons_per_layers))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropOut_rate))
      input_dim = neurons_per_layers
    layers.append(nn.Linear(input_dim , output_dim))

    self.model = nn.Sequential(*layers)

  def forward(self , x):
    return self.model(x)

In [558]:
def objective(trial):
  num_hidden = trial.suggest_int("num_hidden_layer" , 1 , 5)
  neurons_per_layer = trial.suggest_int("neurons_per_layer" , 8 , 128 , step=8)
  epoch = trial.suggest_int("epoch" , 10 , 50 , step=10)
  learning_rate = trial.suggest_float("learning rate" , 1e-5 , 1e-1 , log=True)
  dropOut_rate = trial.suggest_float("dropOut rate" , 0.0 , 0.5 , step = 0.1 )
  batch_size = trial.suggest_categorical("batch size" ,[16 , 32 , 64 , 128 , 256])
  optimizer_name = trial.suggest_categorical('suggest_categorical' , ["Adam" , "SGD" , "RMSprop"])
  weight_decay = trial.suggest_float("weight decay" , 1e-5 , 1e-3 , log=True)
  # model init
  input_dim = 784
  output_dim = 10

  # dataset_test = customDataset(x_test , y_test)
  dataLoaded_test = DataLoader(dataset_test , batch_size=batch_size , shuffle= True)

  model = MyNN(input_dim , output_dim ,num_hidden , neurons_per_layer , dropOut_rate )
  model.to(device)


  loss_fn = nn.CrossEntropyLoss()
  if optimizer_name == "Adam":
    optimizer = optim.Adam(model.parameters() , lr=learning_rate , weight_decay=weight_decay)
  elif optimizer_name == "SGD":
    optimizer = optim.SGD(model.parameters() , lr=learning_rate , weight_decay=weight_decay)
  else :
    optimizer = optim.RMSprop(model.parameters() , lr=learning_rate , weight_decay=weight_decay)
  for i in range(epoch):
    for value , target in dataLoaded_train:
      value = value.to(device)
      target = target.to(device)
      optimizer.zero_grad()

      y_pred = model(value.float())
      loss = loss_fn(y_pred , target)

      loss.backward()
      optimizer.step()
  model.eval()
  total_train = 0
  correct_train = 0
  with torch.no_grad():
    for value , target in dataLoaded_train:

      value = value.to(device)
      target = target.to(device)

      outputs = model(value.float())
      _ , predict = torch.max(outputs , 1)
      total_train += target.shape[0]
      correct_train = correct_train + (predict == target).sum().item()
  accuracy = (correct_train/total_train)*100
  return accuracy

In [559]:
import optuna
study = optuna.create_study(direction="maximize")

[I 2026-07-06 15:41:26,942] A new study created in memory with name: no-name-fc4b27ce-c86c-4b76-b60c-60031d4416f0


In [560]:
study.optimize(objective , n_trials=10)

[I 2026-07-06 15:42:09,608] Trial 0 finished with value: 91.32333333333334 and parameters: {'num_hidden_layer': 1, 'neurons_per_layer': 64, 'epoch': 40, 'learning rate': 0.011469800287167162, 'dropOut rate': 0.0, 'batch size': 128, 'suggest_categorical': 'SGD', 'weight decay': 1.8264653885004644e-05}. Best is trial 0 with value: 91.32333333333334.
[I 2026-07-06 15:42:33,565] Trial 1 finished with value: 79.65 and parameters: {'num_hidden_layer': 1, 'neurons_per_layer': 80, 'epoch': 20, 'learning rate': 0.03873574895063854, 'dropOut rate': 0.5, 'batch size': 256, 'suggest_categorical': 'RMSprop', 'weight decay': 1.8297218572949532e-05}. Best is trial 0 with value: 91.32333333333334.
[I 2026-07-06 15:43:49,357] Trial 2 finished with value: 79.93 and parameters: {'num_hidden_layer': 4, 'neurons_per_layer': 32, 'epoch': 40, 'learning rate': 1.6855742428757632e-05, 'dropOut rate': 0.4, 'batch size': 64, 'suggest_categorical': 'Adam', 'weight decay': 0.0005870983460724573}. Best is trial 0 w

Conclusion

1. Built a neural network from scratch using PyTorch
2.  Learned how DataLoader works.
3. Improved accuracy using Dropout.
4. Used Optuna to tune hyperparameters.